[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/17-mixture-of-experts.ipynb)

# Module 17: Mixture of Experts

Companion notebook to [Module 17](https://transformers-and-math.drmhse.com/modules/17-mixture-of-experts) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — A mixture-of-experts layer (sections 17.1–17.3)

In [ ]:
class MoE(nn.Module):
    def __init__(self, d=64, n_experts=8, k=2, d_ff=128, shared=True):
        super().__init__()
        self.k = k
        self.router = nn.Linear(d, n_experts, bias=False)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d)) for _ in range(n_experts)])
        self.shared = nn.Sequential(nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d)) if shared else None
    def forward(self, x):                       # x: (tokens, d)
        probs = torch.softmax(self.router(x), -1)
        top_p, top_i = probs.topk(self.k, -1)
        if self.k > 1:
            top_p = top_p / top_p.sum(-1, keepdim=True)  # Mixtral-style renormalization
        # With top-1, renormalizing would make the weight exactly 1 and the router would get no
        # gradient at all; Switch Transformer keeps the raw probability instead.
        y = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            rows, slot = (top_i == e).nonzero(as_tuple=True)
            if len(rows):
                y[rows] += top_p[rows, slot, None] * expert(x[rows])
        if self.shared is not None:
            y = y + self.shared(x)
        # load-balancing loss (section 17.4)
        f = F.one_hot(top_i[:, 0], len(self.experts)).float().mean(0)
        aux = len(self.experts) * (f * probs.mean(0)).sum()
        return y, aux, top_i

moe = MoE()
y, aux, chosen = moe(torch.randn(256, 64))
print('output', tuple(y.shape), '| aux loss', round(aux.item(), 3), '| tokens per expert', torch.bincount(chosen.flatten(), minlength=8).tolist())
print('ways to choose 8 of 64 experts:', f'{math.comb(64, 8):,}', ' vs 2 of 16:', math.comb(16, 2))

## Part 2 — Router collapse, and the loss that prevents it (section 17.4)
Train a top-1 MoE layer to imitate a fixed random network, with and without the auxiliary loss. Experts that get more tokens get more updates and improve faster, and the router learns to send them still more: the feedback loop of section 17.4. The router starts out fairly confident (its weights are scaled up), which speeds the loop up.

In [ ]:
def run(alpha, steps=1500, seed=0):
    torch.manual_seed(seed)
    m = MoE(k=1, shared=False)
    with torch.no_grad():
        m.router.weight.mul_(10)
    target = nn.Sequential(nn.Linear(64, 128), nn.GELU(), nn.Linear(128, 64))
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)
    for _ in range(steps):
        x = torch.randn(256, 64)
        y, aux, top = m(x)
        loss = F.mse_loss(y, target(x).detach()) + alpha * aux
        opt.zero_grad(); loss.backward(); opt.step()
    share = torch.bincount(top[:, 0], minlength=8).float() / 256
    return share

for alpha in (0.0, 0.1):
    share = run(alpha)
    idle = int((share < 0.02).sum())
    print(f'aux weight {alpha}: share per expert {[round(s, 2) for s in share.tolist()]}  ({idle} experts nearly idle)')

## Part 3 — Capacity and dropped tokens (section 17.5)

In [ ]:
tokens, K, E, gamma = 4096, 8, 64, 1.25
C = math.ceil(gamma * K * tokens / E)
print(f'capacity per expert: {C} slots')
loads = torch.bincount(torch.randint(0, E, (tokens * K,)), minlength=E)     # even-ish random routing
print('tokens dropped with random routing:', int((loads - C).clamp(min=0).sum()), f'of {tokens * K}')

## Your turn
1. In Part 2, try `seed=1` and `seed=2`. Is the number of idle experts without the auxiliary loss consistent?
2. In Part 3, skew the routing (for example, send 30% of choices to expert 0) and count the dropped tokens.